<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Demo File for Pick and Place
  </span>
</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization [Connect + PowerON]
</span>

In [6]:
# robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
# robotConfig = ysw.setRobotConfig(ysw.MODE_LOCAL_COMPUTER, ysw.MODEL_A)      # For Real Robot + Laptop (RBY1-Hotspot)
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [8]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [7]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:gray; font-weight:bold; font-size:50px;">
Get Forward Kinematics
</span>

In [4]:
# [Link]:     "link_right_arm_6" 
# -> [Joint]: "tool_right"
# -> [Link]:  "ee_right" 
# -> [Joint]: "gripper_finger_r1"
# -> [Link]:  "ee_finger_r1"    (r1이 ee_right 기준으로 +x 방향, 즉 정면에서 볼때 가까운 방향의 그리퍼임)

# ysw.get_FK(robot, reference_link="base", target_link="link_torso_5")
# ysw.get_FK(robot, reference_link="base", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")
ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")

*************** [FK of 'ee_finger_r1' from 'link_head_2' Starts] ***************

[Result]

T:
[[ 0.19674  0.17101 -0.96543  0.54588]
 [ 0.262    0.93969  0.21984 -0.3291 ]
 [ 0.9448  -0.2962   0.14007 -0.10022]
 [ 0.       0.       0.       1.     ]]


position [m]: [ 0.54588 -0.3291  -0.10022]
rpy [rad]: [-1.12906 -1.23698  0.92671]
rpy [deg]: [-64.69032 -70.87389  53.09641]

**************** [FK of 'ee_finger_r1' from 'link_head_2' Ends] ****************





{'T': array([[ 0.19674,  0.17101, -0.96543,  0.54588],
        [ 0.262  ,  0.93969,  0.21984, -0.3291 ],
        [ 0.9448 , -0.2962 ,  0.14007, -0.10022],
        [ 0.     ,  0.     ,  0.     ,  1.     ]]),
 'position': array([ 0.54588, -0.3291 , -0.10022]),
 'rotation': array([[ 0.19674,  0.17101, -0.96543],
        [ 0.262  ,  0.93969,  0.21984],
        [ 0.9448 , -0.2962 ,  0.14007]]),
 'rpy': array([-1.12906, -1.23698,  0.92671])}

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Position Control based on D435i frame
</span>

In [18]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)


### FK example
# Execute: ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")
# Result: T:
            # [[ 0.19674  0.17101 -0.96543  0.54588]
            # [ 0.262    0.93969  0.21984 -0.3291 ]
            # [ 0.9448  -0.2962   0.14007 -0.10022]
            # [ 0.       0.       0.       1.     ]]
            
            # position [m]: [ 0.54588 -0.3291  -0.10022]
            # rpy [rad]: [-1.12906 -1.23698  0.92671]
            # rpy [deg]: [-64.69032 -70.87389  53.09641]
            
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")            
pos = fk_result["position"]
rpy = fk_result["rpy"]
            
R1 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
R2 = ysw.rotMat_x(90.0*ysw.DEG2RAD)     # "link_head_2" frame 기준으로 x축 +90도 회전
R = R1 @ R2
rpy_changed = ysw.rotMat_to_rpy(R)


### Cartesian Position Control example (Whole-Body version)
cartesian_position_targets_sample = [    
                                {
                                    "reference_link": "link_head_2",
                                    "target_link": "ee_finger_r1",
                                    "pos": pos + [0.1, 0.0, 0],                     # 현재 위치는 유지
                                    "rpy": rpy,                     # 자세만 link_head_2 기준으로 변경
                                },                 
                            ]


joint_targets_sample = {
                        "right_arm_6": 0.0 * ysw.DEG2RAD,
                        "left_arm_6": 0.0 * ysw.DEG2RAD,
                       }

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    get_return=True
)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************



*************** [FK of 'ee_finger_r1' from 'link_head_2' Starts] ***************

[Result]

T:
[[ 0.19679  0.17094 -0.96543  0.55015]
 [ 0.26219  0.93965  0.21982 -0.32316]
 [ 0.94474 -0.29638  0.1401  -0.14841]
 [ 0.       0.       0.       1.     ]]


position [m]: [ 0.55015 -0.32316 -0.14841]
rpy [rad]: [-1.12922 -1.2368   0.92692]
rpy [deg]: [-64.69976 -70.8632   53.10834]

**************** [FK of 'ee_finger_r1' from 'link_head_2' Ends] ****************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_r1', pos=[ 0.65015 -0.32316 -0.14841], rpy=[-1.12922 -1.2368   0.92692]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
[Test] 3D information of colored object
</span>

In [4]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] center_3d=[-0.234  0.149  0.725], bbox_3d_size=[0.042 0.056 0.047]
[object 1] center_3d=[0.226 0.133 0.759], bbox_3d_size=[0.064 0.043 0.043]
[object 2] center_3d=[0.263 0.007 0.911], bbox_3d_size=[0.042 0.036 0.042]
[object 3] center_3d=[-0.019  0.116  0.787], bbox_3d_size=[0.154 0.243 0.226]
[object 4] center_3d=[ 0.391 -0.603  1.884], bbox_3d_size=[0.042 0.196 0.14 ]


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] center_3d=[-0.24   0.154  0.728], bbox_3d_size=[0.045 0.057 0.074]
[object 1] center_3d=[0.227 0.134 0.758], bbox_3d_size=[0.067 0.042 0.038]
[object 2] center_3d=[-0.018  0.12   0.783], bbox_3d_size=[0.166 0.245 0.228]
[object 3] center_3d=[ 0.267 -0.012  0.903], bbox_3d_size=[0.058 0.073 0.096]
[object 4] center_3d=[ 0.392 -0.584  1.872], bbox_3d_size=[0.047 0.194 0.143]
[object 0] center_3d=[-0.238  0.151  0.725], bbox_3d_size=[0.041 0.057 0.071]
[object 1] center_3d=[0.225 0.134 0.759], bbox_3d_size=[0.068 0.044 0.054]
[object 2] center_3d=[-0.015  0.125  0.776], bbox_3d_size=[0.166 0.248 0.22 ]
[object 3] center_3d=[ 0.27  -0.013  0.898], bbox_3d_size=[0.059 0.074 0.115]
[object 4] center_3d=[ 0.394 -0.586  1.884], bbox_3d_size=[0.047 0.197 0.134]
[object 0] center_3d=[-0.238  0.15   0.725], bbox_3d_size=[0.05  0.058 0.036]
[object 1] center_3d=[0.228 0.135 0.758], bbox_3d_size=[0.068 0.044 0.034]
[object 2] center_3d=[-0.018  0.122  0.782], bbox_3d_size=[0.162 0.248 0.

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Detect the largest object(box) and the second object
</span>

In [4]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640, 480), fps=30)
d435i.initialize()

record_duration = 5.0  # [sec]

largest_samples = []        # Box
second_samples = []         # Object

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300
        )

        if len(detected_objects) == 0:
            continue

        detected_objects_sorted = sorted(
            detected_objects,
            key=lambda obj: obj["area_2d"],
            reverse=True
        )

        largest_obj = detected_objects_sorted[0]

        largest_samples.append({
            "area_2d": float(largest_obj["area_2d"]),
            "pixel_center": np.array(largest_obj["pixel_center"], dtype=np.float64),
            "center_3d": np.array(largest_obj["center_3d"], dtype=np.float64),
            "bbox_3d_size": np.array(largest_obj["bbox_3d_size"], dtype=np.float64),
            "bbox_3d_min": np.array(largest_obj["bbox_3d_min"], dtype=np.float64),
            "bbox_3d_max": np.array(largest_obj["bbox_3d_max"], dtype=np.float64),
        })

        if len(detected_objects_sorted) >= 2:
            second_obj = detected_objects_sorted[1]

            second_samples.append({
                "area_2d": float(second_obj["area_2d"]),
                "pixel_center": np.array(second_obj["pixel_center"], dtype=np.float64),
                "center_3d": np.array(second_obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(second_obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min": np.array(second_obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max": np.array(second_obj["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()


largest_avg = ysw.average_detectColorObjectContour3D(largest_samples)
second_avg = ysw.average_detectColorObjectContour3D(second_samples)


print("\n========== Detection Result ==========")

if largest_avg is not None:
    print("\n[1st Largest Object]")
    print(f"num_samples             : {largest_avg['num_samples']}")
    print(f"area_2d_avg             : {largest_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {largest_avg['area_2d_median']:.1f}")
    print(f"pixel_center_avg        : {np.round(largest_avg['pixel_center_avg'], 1)}")
    print(f"pixel_center_median     : {np.round(largest_avg['pixel_center_median'], 1)}")
    print(f"center_3d_avg           : {np.round(largest_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(largest_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(largest_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(largest_avg['bbox_3d_size_median'], 4)} m")
    print(f"bbox_3d_min_median      : {np.round(largest_avg['bbox_3d_min_median'], 4)} m")
    print(f"bbox_3d_max_median      : {np.round(largest_avg['bbox_3d_max_median'], 4)} m")
else:
    print("\n[1st Largest Object]")
    print("No object detected.")

if second_avg is not None:
    print("\n[2nd Largest Object]")
    print(f"num_samples             : {second_avg['num_samples']}")
    print(f"area_2d_avg             : {second_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {second_avg['area_2d_median']:.1f}")
    print(f"pixel_center_avg        : {np.round(second_avg['pixel_center_avg'], 1)}")
    print(f"pixel_center_median     : {np.round(second_avg['pixel_center_median'], 1)}")
    print(f"center_3d_avg           : {np.round(second_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(second_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(second_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(second_avg['bbox_3d_size_median'], 4)} m")
    print(f"bbox_3d_min_median      : {np.round(second_avg['bbox_3d_min_median'], 4)} m")
    print(f"bbox_3d_max_median      : {np.round(second_avg['bbox_3d_max_median'], 4)} m")
else:
    print("\n[2nd Largest Object]")
    print("No second object detected.")

[INFO] D435i pipeline started

========== Detection Result ==========

[1st Largest Object]
num_samples             : 68
area_2d_avg             : 18960.9
area_2d_median          : 18995.8
pixel_center_avg        : [338. 280.]
pixel_center_median     : [338. 280.]
center_3d_avg           : [0.04   0.0557 0.8495] m
center_3d_median        : [0.04   0.0564 0.85  ] m
bbox_3d_size_avg        : [0.1667 0.2395 0.2089] m
bbox_3d_size_median     : [0.1672 0.2395 0.208 ] m
bbox_3d_min_median      : [-0.0346 -0.0874  0.727 ] m
bbox_3d_max_median      : [0.1325 0.1519 0.936 ] m

[2nd Largest Object]
num_samples             : 68
area_2d_avg             : 1220.4
area_2d_median          : 1278.8
pixel_center_avg        : [335.3 259.1]
pixel_center_median     : [485. 361.]
center_3d_avg           : [-0.1644 -0.1211  1.1757] m
center_3d_median        : [0.216  0.1434 0.7578] m
bbox_3d_size_avg        : [0.1077 0.0724 0.062 ] m
bbox_3d_size_median     : [0.0777 0.0572 0.032 ] m
bbox_3d_min_median      

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Get the 3D information of objects
</span>

In [5]:
box_pos = largest_avg["center_3d_avg"]

object_pos    = second_avg["center_3d_avg"]
object_maxDim = max(second_avg["bbox_3d_size_avg"])


In [8]:
object_position

array([1.0818, 0.1644, 0.1861])

In [10]:
-x_offset

np.float64(-0.0638294234258287)

In [11]:
ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER

-0.03

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move end-effector to object 
</span>

In [6]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)
            
            
x_offset = object_maxDim / 2 + 0.01  
object_position = ysw.convertPos_vision_to_head2(object_pos) + [-x_offset + ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER, 0, 0]
            
if (object_position[1] <= 0):
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
     
    R1 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
    R2 = ysw.rotMat_x(90.0*ysw.DEG2RAD)     # "link_head_2" frame 기준으로 x축 +90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)
    
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": object_position,                  
                                                "rpy": rpy_original,                    
                                            },                 
                                        ]
    
else:
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_l1")    
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
    
    R1 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 x축 -90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)

    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": object_position,                  
                                                "rpy": rpy_original,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    stop_position_tracking_error = 0.001,
    stop_orientation_tracking_error = 0.001,
    get_return=True
)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************



*************** [FK of 'ee_finger_l1' from 'link_head_2' Starts] ***************

[Result]

T:
[[-0.0443 -0.2763 -0.96    0.5657]
 [-0.2993  0.9205 -0.2512  0.3379]
 [ 0.9531  0.2762 -0.1235  0.0032]
 [ 0.      0.      0.      1.    ]]


position [m]: [0.5657 0.3379 0.0032]
rpy [rad]: [ 1.9912 -1.2634 -1.7178]
rpy [deg]: [114.0897 -72.3869 -98.4206]

**************** [FK of 'ee_finger_l1' from 'link_head_2' Ends] ****************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_l1', pos=[1.0818 0.1644 0.1861], rpy=[ 1.9912 -1.2634 -1.7178]
[ERROR] Failed to conduct Cartesian Position Control(Whole-Body). finish_code = FinishCode.Unknown


In [7]:
cartesian_position_targets_sample

[{'reference_link': 'link_head_2',
  'target_link': 'ee_finger_l1',
  'pos': array([1.0818, 0.1644, 0.1861]),
  'rpy': array([ 1.9912, -1.2634, -1.7178])}]